# 03 — EDA: Institutional Absorption Capacity φ_j

Exploratory analysis of **φ_j** — the Institutional Absorption Capacity index.

**Formula**: $\phi_j = \frac{LPI_{c(j)}}{5}$

where:
- $LPI_{c(j)}$ = World Bank Logistics Performance Index score for sector j's country
- Divided by 5 = maximum possible LPI score (absolute normalization)

**Economic meaning**: Even if topological substitutes exist (captured by $1 - V_j$),
a sector can only actually switch suppliers if its institutional environment supports
rapid onboarding of new trade relationships. LPI captures exactly this.

**Structure:**
1. Load and validate phi
2. Country-level LPI distribution
3. phi by country — layer-level institutional capacity
4. phi vs V_j — are they independent?
5. Damping preview: d_j = (1 - V_j) · phi_j
6. Economic validation


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi']        = 120
plt.rcParams['font.size']         = 11
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

from src.network.builder     import build_matrices
from src.dynamics.compute_vj  import compute_Vj
from src.dynamics.compute_phi import compute_phi

print('Imports OK')

## 0. Load data

In [ ]:
Z, F, X, A, B = build_matrices(2018)
print(f'Nodes: {len(A.index)}')

In [ ]:
# Compute V_j (needed for damping preview in Section 5)
V, IRR_node, input_dist = compute_Vj(A, X, Z=Z)

In [ ]:
# Compute phi_j
phi_node, phi_country = compute_phi(
    lpi_path='../data/raw/lpi.xlsx',
    nodes=A.index,
    year=2018
)

In [ ]:
# Master dataframe
nodes     = A.index
countries = nodes.map(lambda x: x.split('_')[0])
sectors   = nodes.map(lambda x: '_'.join(x.split('_')[1:]))

df = pd.DataFrame({
    'node'    : nodes,
    'country' : countries,
    'sector'  : sectors,
    'V_j'     : V.values,
    'phi_j'   : phi_node.values,
    'd_j'     : (1 - V.values) * phi_node.values,
}, index=nodes)

print(f'Master dataframe: {df.shape}')
print(f'\nd_j = (1-V_j) * phi_j statistics:')
print(df['d_j'].describe().to_string())

---
## Section 1 — Country-level LPI Distribution

phi is a **layer-level** property in the multilayer framework —
every sector node within a country layer shares the same phi.

**What to look for:**
- DEU, SWE, SGP at the top
- AGO, SEN, MMR at the bottom
- Distribution shape — is it roughly normal or skewed?

In [ ]:
# ── Full country ranking ───────────────────────────────────────────────────
phi_sorted = phi_country.sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Bar chart — all 81 countries
norm   = (phi_sorted.values - phi_sorted.min()) / (phi_sorted.max() - phi_sorted.min())
colors = plt.cm.RdYlGn(norm)
axes[0].bar(range(len(phi_sorted)), phi_sorted.values,
            color=colors, edgecolor='none', width=0.8)
axes[0].set_xticks(range(len(phi_sorted)))
axes[0].set_xticklabels(phi_sorted.index, rotation=90, fontsize=6)
axes[0].set_ylabel('phi_j (LPI / 5)')
axes[0].set_title(
    'Institutional Absorption Capacity (phi) — All Countries\n'
    'phi_j = LPI / 5 | Green = high capacity | Red = low capacity',
    fontsize=11, fontweight='bold'
)
axes[0].axhline(phi_country.mean(), color='black', linestyle='--',
                linewidth=1, label=f'Mean={phi_country.mean():.3f}')
axes[0].legend(fontsize=8)

# Distribution
axes[1].hist(phi_country.values, bins=20,
             color='steelblue', edgecolor='white')
axes[1].axvline(phi_country.mean(), color='red', linestyle='--',
                linewidth=1.5, label=f'Mean={phi_country.mean():.3f}')
axes[1].axvline(phi_country.median(), color='darkred', linestyle=':',
                linewidth=1.5, label=f'Median={phi_country.median():.3f}')
axes[1].set_xlabel('phi_j (LPI / 5)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of phi across Countries')
axes[1].legend(fontsize=9)

plt.suptitle('phi_j: Institutional Absorption Capacity by Country',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/phi_eda_country_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()

print('phi statistics (country level):')
print(phi_country.describe().to_string())

In [ ]:
# ── Top 20 and Bottom 20 countries ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Top 20
top20 = phi_country.nlargest(20)
axes[0].barh(range(len(top20)), top20.values,
             color=plt.cm.Greens(top20.values / top20.max()),
             edgecolor='white')
axes[0].set_yticks(range(len(top20)))
axes[0].set_yticklabels(top20.index, fontsize=9)
axes[0].invert_yaxis()
axes[0].set_xlabel('phi_j (LPI / 5)')
axes[0].set_title(
    'Top 20 Countries\nHighest Institutional Capacity',
    fontweight='bold'
)
axes[0].grid(axis='x', linestyle='--', alpha=0.4)
# Add value labels
for i, (val, name) in enumerate(zip(top20.values, top20.index)):
    axes[0].text(val + 0.005, i, f'{val:.3f}', va='center', fontsize=8)

# Bottom 20
bot20 = phi_country.nsmallest(20)
axes[1].barh(range(len(bot20)), bot20.values,
             color=plt.cm.Reds_r(bot20.values / bot20.max()),
             edgecolor='white')
axes[1].set_yticks(range(len(bot20)))
axes[1].set_yticklabels(bot20.index, fontsize=9)
axes[1].invert_yaxis()
axes[1].set_xlabel('phi_j (LPI / 5)')
axes[1].set_title(
    'Bottom 20 Countries\nLowest Institutional Capacity',
    fontweight='bold'
)
axes[1].grid(axis='x', linestyle='--', alpha=0.4)
for i, (val, name) in enumerate(zip(bot20.values, bot20.index)):
    axes[1].text(val + 0.005, i, f'{val:.3f}', va='center', fontsize=8)

plt.suptitle('phi_j: Top and Bottom 20 Countries by Institutional Capacity',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/phi_eda_top_bottom.png',
            dpi=150, bbox_inches='tight')
plt.show()

---
## Section 2 — phi as a Layer Property

In the multilayer framework, phi is a **layer-level** attribute —
every node (sector) within a country layer shares the same phi.

This visualization shows how institutional capacity varies across layers.

In [ ]:
# ── phi heatmap: country x sector (layer view) ────────────────────────────
# phi is constant within each row (country) — shows the layer structure
pivot_phi = df.pivot_table(
    index='country', columns='sector',
    values='phi_j', aggfunc='mean'
).fillna(0)

# Sort countries by phi (high to low)
pivot_phi = pivot_phi.loc[
    phi_country.sort_values(ascending=False).index
]

# Keep top 30 sectors by V_j mean (most analytically interesting)
sector_vj = df.groupby('sector')['V_j'].mean().sort_values(ascending=False)
top_sectors = sector_vj.head(25).index
pivot_sub = pivot_phi[top_sectors]

fig, ax = plt.subplots(figsize=(20, 14))
sns.heatmap(
    pivot_sub, ax=ax,
    cmap='RdYlGn',
    vmin=0, vmax=1,
    linewidths=0.3, linecolor='#e0e0e0',
    cbar_kws={'label': 'phi_j (Institutional Capacity)', 'shrink': 0.6},
    xticklabels=True, yticklabels=True,
)
ax.set_title(
    'phi_j: Institutional Capacity — Country × Sector\n'
    'Rows = Countries (layers) | phi is CONSTANT within each row\n'
    'Green = high institutional capacity | Red = low',
    fontsize=12, fontweight='bold', pad=12
)
ax.set_xlabel('Sector (Node)', fontsize=11)
ax.set_ylabel('Country (Layer)', fontsize=11)
ax.tick_params(axis='x', labelsize=7, rotation=90)
ax.tick_params(axis='y', labelsize=7, rotation=0)
plt.tight_layout()
plt.savefig('../outputs/phi_eda_heatmap.png',
            dpi=150, bbox_inches='tight')
plt.show()

print('Note: phi is constant within each row (country layer)')
print('The horizontal bands show the layer-level structure of phi')

---
## Section 3 — phi vs V_j: Are they independent?

**Critical check for thesis**: phi and V_j must be sufficiently independent
to justify treating them as separate factors in d_j = (1-V_j) · phi_j.

If they are highly correlated, the multiplicative form adds nothing
over a single factor model.

**Expected**: low-to-moderate correlation because:
- V_j is determined by NETWORK TOPOLOGY (who supplies whom)
- phi_j is determined by COUNTRY INSTITUTIONS (LPI)
- These are conceptually orthogonal dimensions

In [ ]:
# ── Scatter: phi vs V_j ───────────────────────────────────────────────────
corr_vj_phi = df['V_j'].corr(df['phi_j'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Node level
axes[0].scatter(
    df['phi_j'], df['V_j'],
    alpha=0.1, s=6, color='steelblue'
)
# Highlight extremes
high_v  = df.nlargest(20, 'V_j')
low_phi = df.nsmallest(20, 'phi_j')
axes[0].scatter(high_v['phi_j'], high_v['V_j'],
                color='red', s=40, alpha=0.9,
                zorder=5, label='Top 20 V_j')
axes[0].scatter(low_phi['phi_j'], low_phi['V_j'],
                color='orange', s=40, alpha=0.9,
                zorder=5, label='Bottom 20 phi')
axes[0].set_xlabel('phi_j (Institutional Capacity)')
axes[0].set_ylabel('V_j (Upstream Vulnerability)')
axes[0].set_title(f'phi_j vs V_j (node level)\nPearson r = {corr_vj_phi:.3f}')
axes[0].text(0.05, 0.95, f'r = {corr_vj_phi:.3f}',
             transform=axes[0].transAxes, fontsize=10,
             verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[0].legend(fontsize=8)

# Country level — aggregate V_j by country
country_vj_mean  = df.groupby('country')['V_j'].mean()
country_phi_mean = df.groupby('country')['phi_j'].mean()
corr_country     = country_vj_mean.corr(country_phi_mean)

axes[1].scatter(
    country_phi_mean.values,
    country_vj_mean.values,
    alpha=0.8, s=60, color='darkorange'
)
# Label interesting countries
for country in ['DEU', 'USA', 'CHN', 'AGO', 'TUN', 'UKR', 'RUS', 'BGD', 'SGP']:
    if country in country_phi_mean.index:
        axes[1].annotate(
            country,
            (country_phi_mean[country], country_vj_mean[country]),
            fontsize=8, alpha=0.9
        )
axes[1].set_xlabel('phi (country mean)')
axes[1].set_ylabel('V_j (country mean)')
axes[1].set_title(f'phi vs V_j (country level)\nPearson r = {corr_country:.3f}')
axes[1].text(0.05, 0.95, f'r = {corr_country:.3f}',
             transform=axes[1].transAxes, fontsize=10,
             verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle(
    'Independence Check: phi_j vs V_j\n'
    'Low correlation validates treating them as separate factors in d_j',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('../outputs/phi_eda_vs_vj.png',
            dpi=150, bbox_inches='tight')
plt.show()

print(f'Correlation phi vs V_j (node level)    : {corr_vj_phi:.4f}')
print(f'Correlation phi vs V_j (country level) : {corr_country:.4f}')
print()
if abs(corr_vj_phi) < 0.4:
    print('✓ phi and V_j are sufficiently independent')
    print('  Multiplicative d_j = (1-V_j)·phi_j is justified')
else:
    print('! Moderate correlation detected — discuss in thesis limitations')

---
## Section 4 — The Four Quadrants

Classifying sectors by (V_j, phi_j) position reveals four distinct
economic profiles — this is a key interpretive result for your thesis.

| | High phi (strong institutions) | Low phi (weak institutions) |
|---|---|---|
| **High V_j (fragile topology)** | Structurally exposed but institutionally capable | Doubly fragile — worst case |
| **Low V_j (resilient topology)** | Best case — substitutable AND capable | Resilient topology but institutional bottleneck |

In [ ]:
# ── Four quadrant analysis ────────────────────────────────────────────────
v_median   = df['V_j'].median()
phi_median = df['phi_j'].median()

# Classify each node into quadrant
df['quadrant'] = 'Q1: Low V, High phi (resilient)'
df.loc[(df['V_j'] >= v_median) & (df['phi_j'] >= phi_median),
       'quadrant'] = 'Q2: High V, High phi (exposed but capable)'
df.loc[(df['V_j'] < v_median) & (df['phi_j'] < phi_median),
       'quadrant'] = 'Q3: Low V, Low phi (topology ok, inst. weak)'
df.loc[(df['V_j'] >= v_median) & (df['phi_j'] < phi_median),
       'quadrant'] = 'Q4: High V, Low phi (doubly fragile)'

quadrant_counts = df['quadrant'].value_counts()
print('Sectors per quadrant:')
print(quadrant_counts.to_string())

# Scatter with quadrant colors
colors_q = {
    'Q1: Low V, High phi (resilient)'              : '#27ae60',
    'Q2: High V, High phi (exposed but capable)'   : '#f39c12',
    'Q3: Low V, Low phi (topology ok, inst. weak)' : '#3498db',
    'Q4: High V, Low phi (doubly fragile)'         : '#c0392b',
}

fig, ax = plt.subplots(figsize=(12, 8))
for q, color in colors_q.items():
    mask = df['quadrant'] == q
    ax.scatter(
        df.loc[mask, 'phi_j'],
        df.loc[mask, 'V_j'],
        alpha=0.25, s=10, color=color,
        label=f'{q} (n={mask.sum()})'
    )

# Quadrant dividers
ax.axvline(phi_median, color='black', linestyle='--',
           linewidth=1, alpha=0.5, label=f'phi median={phi_median:.3f}')
ax.axhline(v_median, color='black', linestyle=':',
           linewidth=1, alpha=0.5, label=f'V_j median={v_median:.3f}')

# Quadrant labels
ax.text(0.15, 0.95, 'Doubly\nFragile', transform=ax.transAxes,
        color='#c0392b', fontsize=10, fontweight='bold', ha='center')
ax.text(0.75, 0.95, 'Exposed but\nCapable', transform=ax.transAxes,
        color='#f39c12', fontsize=10, fontweight='bold', ha='center')
ax.text(0.15, 0.05, 'Inst. Weak\nTopology OK', transform=ax.transAxes,
        color='#3498db', fontsize=10, fontweight='bold', ha='center')
ax.text(0.75, 0.05, 'Resilient', transform=ax.transAxes,
        color='#27ae60', fontsize=10, fontweight='bold', ha='center')

ax.set_xlabel('phi_j (Institutional Capacity = LPI/5)', fontsize=11)
ax.set_ylabel('V_j (Upstream Vulnerability)', fontsize=11)
ax.set_title(
    'Four Quadrant Classification: V_j × phi_j\n'
    'Each sector classified by topological exposure and institutional capacity',
    fontsize=12, fontweight='bold'
)
ax.legend(fontsize=8, loc='center right')
plt.tight_layout()
plt.savefig('../outputs/phi_eda_quadrants.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Which countries fall in each quadrant? ────────────────────────────────
print('Country composition of Q4 (Doubly Fragile — High V_j, Low phi):')
q4 = df[df['quadrant'] == 'Q4: High V, Low phi (doubly fragile)']
print(q4.groupby('country').size().sort_values(ascending=False).head(15).to_string())
print()
print('Country composition of Q1 (Resilient — Low V_j, High phi):')
q1 = df[df['quadrant'] == 'Q1: Low V, High phi (resilient)']
print(q1.groupby('country').size().sort_values(ascending=False).head(15).to_string())

---
## Section 5 — Damping Preview: d_j = (1 - V_j) · phi_j

Before running the full damped simulation, preview the distribution
of d_j — the total absorption capacity per sector.

This is what enters the diagonal of matrix D.

In [ ]:
# ── d_j distribution ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# V_j distribution
axes[0].hist(df['V_j'], bins=60, color='#c0392b',
             edgecolor='none', alpha=0.8)
axes[0].set_xlabel('V_j')
axes[0].set_ylabel('Frequency')
axes[0].set_title(f'V_j distribution\nmean={df["V_j"].mean():.3f}')
axes[0].axvline(df['V_j'].mean(), color='black',
                linestyle='--', linewidth=1.5)

# phi_j distribution
axes[1].hist(df['phi_j'], bins=60, color='#2980b9',
             edgecolor='none', alpha=0.8)
axes[1].set_xlabel('phi_j')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'phi_j distribution\nmean={df["phi_j"].mean():.3f}')
axes[1].axvline(df['phi_j'].mean(), color='black',
                linestyle='--', linewidth=1.5)

# d_j distribution
axes[2].hist(df['d_j'], bins=60, color='#27ae60',
             edgecolor='none', alpha=0.8)
axes[2].set_xlabel('d_j = (1-V_j) · phi_j')
axes[2].set_ylabel('Frequency')
axes[2].set_title(f'd_j distribution\nmean={df["d_j"].mean():.3f}')
axes[2].axvline(df['d_j'].mean(), color='black',
                linestyle='--', linewidth=1.5)

plt.suptitle(
    'Damping Components: V_j, phi_j, and d_j = (1-V_j)·phi_j\n'
    'd_j is what enters the diagonal of damping matrix D',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('../outputs/phi_eda_damping_preview.png',
            dpi=150, bbox_inches='tight')
plt.show()

print('d_j = (1 - V_j) * phi_j statistics:')
print(df['d_j'].describe().to_string())

In [ ]:
# ── Top and bottom d_j sectors ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Most absorbed (highest d_j) — these sectors will show least RL change
top_d = df.nlargest(20, 'd_j')[['node','country','sector','V_j','phi_j','d_j']]
axes[0].barh(range(len(top_d)), top_d['d_j'].values,
             color=plt.cm.Greens(top_d['d_j'].values / top_d['d_j'].max()),
             edgecolor='white')
axes[0].set_yticks(range(len(top_d)))
axes[0].set_yticklabels(top_d['node'].values, fontsize=8)
axes[0].invert_yaxis()
axes[0].set_xlabel('d_j = (1-V_j)·phi_j')
axes[0].set_title('Top 20 Highest Absorption\n(most damped sectors)',
                  fontweight='bold')
axes[0].grid(axis='x', linestyle='--', alpha=0.4)

# Least absorbed (lowest d_j) — these sectors will show most RL
bot_d = df.nsmallest(20, 'd_j')[['node','country','sector','V_j','phi_j','d_j']]
axes[1].barh(range(len(bot_d)), bot_d['d_j'].values,
             color=plt.cm.Reds_r(bot_d['d_j'].values / bot_d['d_j'].max()),
             edgecolor='white')
axes[1].set_yticks(range(len(bot_d)))
axes[1].set_yticklabels(bot_d['node'].values, fontsize=8)
axes[1].invert_yaxis()
axes[1].set_xlabel('d_j = (1-V_j)·phi_j')
axes[1].set_title('Bottom 20 Lowest Absorption\n(least damped — most fragile)',
                  fontweight='bold')
axes[1].grid(axis='x', linestyle='--', alpha=0.4)

plt.suptitle('Damping Coefficient d_j per Sector',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/phi_eda_dj_ranking.png',
            dpi=150, bbox_inches='tight')
plt.show()

print('Top 20 most absorbed sectors:')
print(top_d.to_string(index=False))
print()
print('Bottom 20 least absorbed (most fragile) sectors:')
print(bot_d.to_string(index=False))

---
## Section 6 — Economic Validation

**Expected pattern for d_j:**
- High d_j: developed economies with diversified supplier bases (DEU, USA, SGP)
- Low d_j: developing economies dependent on concentrated global products (AGO, TUN, BGD)
- The doubly fragile (Q4) sectors should have lowest d_j

In [ ]:
# ── d_j heatmap: country x sector ─────────────────────────────────────────
pivot_d = df.pivot_table(
    index='country', columns='sector',
    values='d_j', aggfunc='mean'
).fillna(0)

# Sort by mean d_j (most absorbed at top)
country_d = df.groupby('country')['d_j'].mean().sort_values(ascending=False)
sector_d  = df.groupby('sector')['d_j'].mean().sort_values(ascending=False)

top_ctry = country_d.head(35).index
top_sec  = sector_d.head(25).index
pivot_sub = pivot_d.loc[top_ctry, top_sec]

fig, ax = plt.subplots(figsize=(20, 12))
sns.heatmap(
    pivot_sub, ax=ax,
    cmap='RdYlGn',
    vmin=0, vmax=pivot_d.values.max(),
    linewidths=0.3, linecolor='#e0e0e0',
    cbar_kws={'label': 'd_j = (1-V_j)·phi_j', 'shrink': 0.6},
    xticklabels=True, yticklabels=True,
)
ax.set_title(
    'Damping Coefficient d_j: Country × Sector\n'
    'Green = high absorption | Red = low absorption (fragile)\n'
    'd_j = (1 - V_j) · phi_j',
    fontsize=12, fontweight='bold', pad=12
)
ax.set_xlabel('Sector (Node)', fontsize=11)
ax.set_ylabel('Country (Layer)', fontsize=11)
ax.tick_params(axis='x', labelsize=7, rotation=90)
ax.tick_params(axis='y', labelsize=7, rotation=0)
plt.tight_layout()
plt.savefig('../outputs/phi_eda_dj_heatmap.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary table for thesis ───────────────────────────────────────────────
print('=' * 60)
print('phi_j and d_j EDA SUMMARY')
print('=' * 60)
print(f'phi_j range    : [{phi_node.min():.4f}, {phi_node.max():.4f}]')
print(f'phi_j mean     : {phi_node.mean():.4f}')
print(f'phi_j median   : {phi_node.median():.4f}')
print()
print(f'd_j range      : [{df["d_j"].min():.4f}, {df["d_j"].max():.4f}]')
print(f'd_j mean       : {df["d_j"].mean():.4f}')
print(f'd_j median     : {df["d_j"].median():.4f}')
print()
print(f'Correlation phi vs V_j  : {corr_vj_phi:.4f}')
print()
print(f'Quadrant counts:')
print(quadrant_counts.to_string())
print()
print(f'Highest d_j country : {country_d.idxmax()} ({country_d.max():.4f})')
print(f'Lowest d_j country  : {country_d.idxmin()} ({country_d.min():.4f})')
print()
print(f'Highest d_j node : {df["d_j"].idxmax()} ({df["d_j"].max():.4f})')
print(f'Lowest d_j node  : {df["d_j"].idxmin()} ({df["d_j"].min():.4f})')